# justoliunet_opt / justoliunet_bn_opt on the FPGA

The optimized 1D-Justo-LiuNet (or its BatchNorm version): the whole capture in one start,
all of the work in hardware. The kernel reads the raw L1a cube from DDR (two ports,
alternate blocks of 16 pixels), z-scores the kept bands, and runs every pixel's spectrum
through the four conv blocks and the dense layer, a pixel every 18 cycles, in integer fixed
point; it writes each pixel's 3 scores and its class. The answers are checked bit for bit
against the integer model of the kernel (the same as `tools/justoliunet_model.py`'s
`classify_fixed`), on a sample of the pixels, and against the labels.

Cloud / land / sea for every pixel of a HYPSO-2 capture: **0 cloud, 1 land, 2 sea**.

Next to this notebook: the kernel's `.bit` + `.hwh` and `x..._hw.h`, its model's
`..._int8.npz` and `justoliunet_opt_formats.hpp`, and an image zip made on the PC with
`python3 tools/justounetsimple_image.py prepare ... --out aegean_unet` and `fpga_zip aegean_unet`
(the same image as for the other kernels). Run the cells from the top.

The image zip can also go in the folder above this one, once for every design: the notebook looks there when it is not next to it, unpacks it there, and writes its own results next to itself.

## 1. Load the bitstream

In [ ]:
import json, re, time, zipfile
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from pynq import Overlay, allocate, Clocks

# Which kernel: its register map, x<NAME>_hw.h, lies next to its NAME.bit.
header_file = next(Path('.').glob('x*_hw.h'))
NAME = header_file.name[1:-len('_hw.h')]
MODEL = NAME[:-len('_opt')]                     # justoliunet or justoliunet_bn: its weights
overlay = Overlay(f'{NAME}.bit')                 # also sets the FPGA clock from the .hwh
kernel = overlay.kernel                         # the HLS block, named 'kernel' in the block design
MHZ = Clocks.fclk0_mhz                          # the kernel's clock, as the overlay set it

header = header_file.read_text()
def register(name):
    return int(re.search(rf'_ADDR_{name}\s+(0x[0-9a-fA-F]+)', header).group(1), 16)
AP_CTRL, HEIGHT, WIDTH, DOUT = register('AP_CTRL'), register('HEIGHT_DATA'), register('WIDTH_DATA'), register('DOUT_DATA')
DINS = [int(a, 16) for a in re.findall(r'_ADDR_DIN\d+_DATA\s+(0x[0-9a-fA-F]+)', header)]   # din0, din1
CYCLES = register('CYCLES_DATA')                # the kernel's own count of its cycles

# The kernel's formats (justoliunet_opt.hpp): in, the raw L1a cube as captured
# (H x W x 120 uint16); out, H x W x 4 int32: the 3 scores = score * 2^OUT_FRAC, then the
# class (0 cloud, 1 land, 2 sea).
BANDS, LANES = 120, 4

def launch(cube_address, scores_address, height, width):
    # Start the kernel on a height x width cube; both input ports read it.
    for offset, address in [(d, cube_address) for d in DINS] + [(DOUT, scores_address)]:
        kernel.write(offset, address & 0xFFFFFFFF)
        kernel.write(offset + 4, address >> 32)
    kernel.write(HEIGHT, height)
    kernel.write(WIDTH, width)
    kernel.write(AP_CTRL, 1)

def wait(timeout=60):
    deadline = time.monotonic() + timeout
    while not kernel.read(AP_CTRL) & 0x2:      # done
        if time.monotonic() > deadline:
            raise TimeoutError('kernel did not finish')

def counted():
    # The last run's time by the kernel's own count, in seconds.
    return kernel.read(CYCLES) / (MHZ * 1e6)

print(f'{NAME}: {MODEL} weights, {len(DINS)} read ports, {MHZ:.0f} MHz; idle:', bool(kernel.read(AP_CTRL) & 0x4))

## 2. The integer model of the kernel

What the kernel must write, bit for bit, in numpy: the same integers as
`tools/justoliunet_model.py`'s `classify_fixed`, from the model's int8 weights and the
formats header (the z-score constants, the activations' fraction bits).

In [ ]:
formats = Path('justoliunet_opt_formats.hpp').read_text()
def const(name, text=formats):
    return int(re.search(rf'constexpr int {name} = (-?\d+);', text).group(1))
def array(name):
    return np.array(re.search(rf'{name}\[112\] = \{{([^}}]*)\}}', formats).group(1).replace(',', ' ').split(), dtype=np.int64)
IN_FRAC, IN_MAX, PREP_FRAC = const('IN_FRAC'), const('IN_MAX'), const('PREP_FRAC')
ACT_MAX, OUT_FRAC = const('ACT_MAX'), const('OUT_FRAC')
A, B = array('A'), array('B')
model_formats = re.search(rf'namespace {MODEL}_formats \{{(.*?)\}}', formats, re.S).group(1)
ACT_FRACS = [const(f'ACT_FRAC{i}', model_formats) for i in range(1, 5)]
with np.load(f'{MODEL}_int8.npz') as z:
    Q = {name: (z[name].astype(np.int64), int(z[name + '.frac'])) for name in z.files if not name.endswith('.frac')}

def round_shift(s, n):
    return s if n == 0 else (s + (1 << (n - 1))) >> n

def reference_fixed(raw):
    # Raw pixels (..., 120) -> (N, 4) int32: the kernel's scores and class, bit for bit.
    dn = np.asarray(raw).reshape(-1, BANDS).astype(np.int64)[:, 8:118]
    h = np.clip((dn * A[:110] + B[:110]) >> PREP_FRAC, -IN_MAX, IN_MAX)[:, None, :]    # N, C, L
    fa = IN_FRAC
    for i in range(1, 5):
        (w, fw), (b, fb) = Q[f'conv{i}.weight'], Q[f'conv{i}.bias']
        length = h.shape[2] - w.shape[2] + 1
        windows = np.stack([h[:, :, j:j + length] for j in range(w.shape[2])], axis=-1)
        s = np.einsum('nclk,ock->nol', windows, w) + (b << (fa + fw - fb))[None, :, None]
        a = np.clip(round_shift(s, fa + fw - ACT_FRACS[i - 1]), 0, ACT_MAX)
        half = a.shape[2] // 2                                      # an odd last position is dropped
        h, fa = a[:, :, :2 * half].reshape(len(a), a.shape[1], half, 2).max(axis=3), ACT_FRACS[i - 1]
    (w, fw), (b, fb) = Q['fc.weight'], Q['fc.bias']
    s = (h.reshape(len(h), -1) @ w.T + (b << (fa + fw - fb))) << (OUT_FRAC - fa - fw)
    return np.concatenate([s, s.argmax(1)[:, None]], axis=1).astype(np.int32)

def to_float(scores):
    return np.multiply(np.asarray(scores), np.float32(2.0 ** -OUT_FRAC), dtype=np.float32)
print(f'formats: 12-bit input with {IN_FRAC} fraction bits, uint8 activations with {ACT_FRACS} fraction bits')

## 3. Load the image into DDR

The image zip is on the SD card, next to this notebook; it is unpacked there (only when the
zip is newer than what is unpacked). The raw cube goes from the SD card into DDR that the
FPGA can read (PYNQ's CMA, shared by every running notebook: shut down the ones you are not
using). If the whole cube does not fit, it goes to the FPGA in strips of 128 rows, two strip
buffers taking turns.

In [ ]:
# The image: IMAGE.zip or the folder IMAGE/, next to this notebook or in the folder above
# it. Above is shared by every design's folder: upload the image there once. A zip is
# unpacked next to itself, and again only when it is newer than what was unpacked.
IMAGE = 'aegean_unet'
place = next((d for d in (Path('.'), Path('..'))
              if (d / f'{IMAGE}.zip').exists() or (d / IMAGE / 'cube.npy').exists()), None)
if place is None:
    raise FileNotFoundError(f'no {IMAGE}.zip or {IMAGE}/ in {Path(".").resolve()} or the folder above it')
folder, archive = place / IMAGE, place / f'{IMAGE}.zip'
if archive.exists() and (not (folder / 'cube.npy').exists()
                         or (folder / 'cube.npy').stat().st_mtime < archive.stat().st_mtime):
    with zipfile.ZipFile(archive) as z:
        z.extractall(place)
print(f'image: {folder.resolve()}')
t0 = time.monotonic()
cube = np.load(folder / 'cube.npy', mmap_mode='r')     # read from the SD card as it is used
if cube.dtype != np.uint16:                           # an image folder from before: counts as floats
    cube = np.rint(cube).astype(np.uint16)
meta = json.loads((folder / 'meta.json').read_text())
H, W, _ = cube.shape
PIXELS = H * W

scores_out = allocate(shape=(H, W, LANES), dtype=np.int32)
STRIP = 128                                           # rows per strip, if the cube does not fit
try:
    cube_in = allocate(shape=cube.shape, dtype=np.uint16)
    cube_in[:] = cube
    cube_in.flush()                                   # into DDR, not just the CPU cache
    strips = None
except Exception as error:                            # not enough (contiguous) CMA
    cube_in = None
    cube = np.ascontiguousarray(cube)                 # all of it in memory, for the strips
    strips = [allocate(shape=(STRIP, W, BANDS), dtype=np.uint16) for _ in range(2)]
    print(f'the whole cube does not fit in CMA ({error}): strips of {STRIP} rows instead')
print(f'{H} x {W} pixels, loaded in {time.monotonic() - t0:.1f} s')

## 4. Self-test: the first pixels against the integer model

In [ ]:
# The first pixels on their own, as an image of one row, in their own buffers.
n = min(PIXELS, 256)
test_in = allocate(shape=(1, n, BANDS), dtype=np.uint16)
test_out = allocate(shape=(1, n, LANES), dtype=np.int32)
test_in[:] = cube.reshape(-1, BANDS)[:n].reshape(1, n, BANDS)
test_in.flush()
launch(test_in.physical_address, test_out.physical_address, 1, n)
wait()
test_out.invalidate()
differ = int((np.asarray(test_out).reshape(n, LANES) != reference_fixed(test_in)).any(axis=1).sum())
print('PASS' if differ == 0 else 'FAIL', f'{differ} of {n} pixels differ from the integer model')

## 5. The whole image

**Wall clock**: from calling `classify` until the ARM can read the scores and classes.
**FPGA**: from the start bit until the done bit, as the ARM saw it; **by its own count**: the
kernel's own count of its cycles, without the ARM's register writes and polling.

In [ ]:
def classify(timeout=60):
    # The loaded cube -> (H x W x 4 int32: scores and class, seconds the FPGA ran as the ARM
    # saw it, seconds by the kernel's own count).
    if strips is None:
        t = time.perf_counter()
        launch(cube_in.physical_address, scores_out.physical_address, H, W)
        wait(timeout)
        return np.asarray(scores_out), time.perf_counter() - t, counted()
    # Double-buffered strips: the ARM copies strip s + 1 while the FPGA does strip s.
    def fill(s):
        rows = cube[s * STRIP:(s + 1) * STRIP]
        strips[s % 2][:len(rows)] = rows
        strips[s % 2].flush()
        return len(rows)
    count, fpga, own = -(-H // STRIP), 0.0, 0.0
    rows = fill(0)
    for s in range(count):
        t = time.perf_counter()
        launch(strips[s % 2].physical_address, scores_out.physical_address + s * STRIP * W * LANES * 4, rows, W)
        next_rows = fill(s + 1) if s + 1 < count else 0
        wait(timeout)
        fpga += time.perf_counter() - t               # with the copy, if that took longer
        own += counted()                              # the kernel alone, without the copy
        rows = next_rows
    return np.asarray(scores_out), fpga, own

t = time.perf_counter()
raw_scores, fpga, own = classify()
scores_out.invalidate()                               # read what the kernel wrote
raw_scores = np.asarray(scores_out)
wall = time.perf_counter() - t
classes = raw_scores[..., 3]
print(f'wall clock:        {1000 * wall:8.1f} ms for {H} x {W} pixels')
print(f'FPGA:              {1000 * fpga:8.1f} ms' + ('' if strips is None else f' (in strips of {STRIP} rows)'))
print(f'by its own count:  {1000 * own:8.1f} ms, {own * MHZ * 1e6 / PIXELS:.1f} cycles per pixel at {MHZ:.0f} MHz')
np.save(f'{IMAGE}_fpga_scores.npy', to_float(raw_scores[..., :3]))   # next to this notebook: the image folder is shared

## 6. Check it: against the integer model and against the labels

The integer model in numpy on the ARM is slow (57 000 multiply-adds a pixel), so it checks a
fixed sample of up to 20 000 pixels spread over the image; the labels check every pixel.

In [ ]:
sample = np.unique(np.linspace(0, PIXELS - 1, min(PIXELS, 20000)).astype(np.int64))
t = time.perf_counter()
want = reference_fixed(cube.reshape(-1, BANDS)[sample])
got = raw_scores.reshape(-1, LANES)[sample]
close = (got == want).all(axis=1)
print(f'bit-exact against the integer model: {close.mean():.2%} of {len(sample)} pixels '
      f'({time.perf_counter() - t:.1f} s of numpy)')
labels = np.load(folder / 'labels.npy') if (folder / 'labels.npy').exists() else None
if labels is not None:
    known = labels < 3
    print(f'accuracy against the labels: {(classes[known] == labels[known]).mean():.2%}')
    for c, name in enumerate(('cloud', 'land', 'sea')):
        union = ((classes == c) | (labels == c))[known].sum()
        iou = ((classes == c) & (labels == c))[known].sum() / union if union else float('nan')
        print(f'  IoU {name}: {iou:.3f}')

In [ ]:
COLORS = np.array([[235, 235, 235], [60, 140, 60], [30, 90, 180], [128, 128, 128]], dtype=np.uint8)
rgb = cube[:, :, meta['rgb_bands']].astype(np.float64)
lo, hi = np.percentile(rgb, [2, 98])
panels = {'picture': np.clip((rgb - lo) / (hi - lo), 0, 1),
          'FPGA classes': COLORS[np.clip(classes, 0, 3)]}
if labels is not None:
    panels['labels'] = COLORS[np.minimum(labels, 3)]
    wrong = np.full(labels.shape + (3,), 255, dtype=np.uint8)   # white: same class as the label
    wrong[~known] = 128                                         # gray: unlabeled
    wrong[known & (classes != labels)] = (220, 30, 30)          # red: a different class
    panels['differs from labels'] = wrong
fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 4))
for ax, (title, img) in zip(axes, panels.items()):
    ax.imshow(img)
    ax.set_title(title)
    ax.axis('off')
plt.show()     # classes: cloud white, land green, sea blue, gray = unlabeled

## 7. Speed: the FPGA alone

The kernel again on the cube already in DDR (in strips: on one strip), best of 5. The kernel
was designed for 18 cycles a pixel (justoliunet_opt.hpp): its own count says how close it is.

In [ ]:
buffer, rows = (cube_in, H) if strips is None else (strips[0], min(STRIP, H))
if strips is not None:
    strips[0][:rows] = cube[:rows]
    strips[0].flush()
runs, own_runs = [], []
for _ in range(5):
    t = time.perf_counter()
    launch(buffer.physical_address, scores_out.physical_address, rows, W)
    wait()
    runs.append(time.perf_counter() - t)
    own_runs.append(counted())
best, own_best = min(runs), min(own_runs)
print(f'FPGA alone:       {1000 * best:.1f} ms for {rows} x {W} pixels, {1e9 * best / (rows * W):.0f} ns per pixel')
print(f'by its own count: {1000 * own_best:.1f} ms, {own_best * MHZ * 1e6 / (rows * W):.2f} cycles per pixel at {MHZ:.0f} MHz')
print(f'the whole image: about {1000 * own_best * H / rows:.1f} ms on the FPGA, {1000 * wall:.1f} ms wall clock')